In [1]:
from features import *
from fault_injection import *
import numpy as np
import pandas as pd


#### Load dataset (this will take a while)

In [3]:
import glob
import pandas as pd
from fault_injection import generate_faulty_dataset

DATASET_PATH = "./data/data/UNIT_0001_RUN_033"
csv_files = glob.glob(f"{DATASET_PATH}/*.csv")

rows = []
for entry in generate_faulty_dataset(csv_files, prediction_horizon_sec=60.0):
    labels = entry["labels"]
    rows.append({
        "fault_type": entry["fault_type"],
        "severity":   round(float(entry["severity"]), 4),
        "fs":         round(entry["fs"], 1),
        "is_imu":     entry["is_imu"],
        "n_samples":  len(labels),
        "n_healthy":  int((labels == 0).sum()),
        "n_prefault": int((labels == 1).sum()),
        "n_fault":    int((labels == 2).sum()),
    })

pd.DataFrame(rows)

KeyboardInterrupt: 

In [ ]:
import os
import glob

from features import *
from pipeline import run_pipeline

DATASET_PATH = "./data/data/UNIT_0001_RUN_033"
csv_files = glob.glob(f"{DATASET_PATH}/*.csv")

# Discover available sensors before committing to a run
available_sensors = sorted({os.path.splitext(os.path.basename(f))[0] for f in csv_files})
print("Available sensors:")
for s in available_sensors:
    print(f"  {s}")

Available sensors:
  accelerometer
  current
  gyroscope
  magnetometer
  microphone
  photodiode
  pressure
  temperature
  vibration


#### Run pipeline — single-sensor mode\n\nSet `SENSOR_ID` to one of the names printed above to train on that sensor only.\nLeave it as `None` to process all sensors together.

In [ ]:
# Set to a sensor name (e.g. "temperature") for single-sensor training, or None for all sensors
SENSOR_ID = "temperature"

(
    X_imu, X_scalar,
    y_imu, y_scalar,
    scaler_imu, scaler_scalar,
    sensor_imu, sensor_scalar,
    feature_names_imu, feature_names_scalar,
) = run_pipeline(
    csv_files,
    prediction_horizon_sec=20.0,
    n_workers=os.cpu_count() - 1,
    sensor_filter=SENSOR_ID,
)

if X_scalar is not None:
    print("\nScalar feature columns:")
    for i, name in enumerate(feature_names_scalar):
        print(f"  [{i:>2}] {name}")

if X_imu is not None:
    print("\nIMU feature columns (first 10):")
    for i, name in enumerate(feature_names_imu[:10]):
        print(f"  [{i:>2}] {name}")
    print(f"  ... ({len(feature_names_imu)} total)")

Filtered to 1 file(s) for sensor 'temperature'
Total jobs: 3


/Users/samkorostov/Projects/project-shield/shield-model/features.py:82: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  sp_stats.skew(w),
/Users/samkorostov/Projects/project-shield/shield-model/features.py:82: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  sp_stats.skew(w),
/Users/samkorostov/Projects/project-shield/shield-model/features.py:83: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  sp_stats.kurtosis(w),
/Users/samkorostov/Projects/project-shield/shield-model/features.py:83: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly id

Processed entry 3 (sensor=temperature, fault=drift)
Processed entry 1 (sensor=temperature, fault=healthy)
Processed entry 2 (sensor=temperature, fault=bias_offset)
Scalar: X=(126468, 22)  y=(126468,)  sensors=['temperature']  labels=(array([0, 1, 2], dtype=int8), array([121827,     31,   4610]))

Scalar feature columns:
  [ 0] time__mean
  [ 1] time__var
  [ 2] time__rms
  [ 3] time__skew
  [ 4] time__kurtosis
  [ 5] time__zero_crossing_rate
  [ 6] freq__band_energy_0_10hz
  [ 7] freq__band_energy_10_100hz
  [ 8] freq__band_energy_100_500hz
  [ 9] freq__spectral_centroid
  [10] freq__spectral_flatness
  [11] stability__mean_squared_diff
  [12] modwt__detail_energy_L1
  [13] modwt__detail_variance_L1
  [14] modwt__detail_energy_L2
  [15] modwt__detail_variance_L2
  [16] modwt__detail_energy_L3
  [17] modwt__detail_variance_L3
  [18] modwt__detail_energy_L4
  [19] modwt__detail_variance_L4
  [20] modwt__approx_energy_L4
  [21] modwt__approx_variance_L4


#### Save processed feature vectors and scalers

In [ ]:
import joblib
import json

if X_imu is not None:
    np.save("X_imu.npy", X_imu.astype(np.float32))
    np.save("y_imu.npy", y_imu.astype(np.int8))
    np.save("sensor_imu.npy", sensor_imu)
    joblib.dump(scaler_imu, "scaler_imu.joblib")
    with open("feature_names_imu.json", "w") as f:
        json.dump(feature_names_imu, f, indent=2)

if X_scalar is not None:
    np.save("X_scalar.npy", X_scalar.astype(np.float32))
    np.save("y_scalar.npy", y_scalar.astype(np.int8))
    np.save("sensor_scalar.npy", sensor_scalar)
    joblib.dump(scaler_scalar, "scaler_scalar.joblib")
    with open("feature_names_scalar.json", "w") as f:
        json.dump(feature_names_scalar, f, indent=2)

In [ ]:
import glob
import numpy as np

# Set to a sensor name to train on one sensor, or None for all sensors
SENSOR = "temperature"

pattern = f"processed_windows/{SENSOR}" if SENSOR else "processed_windows/*"
X_files = sorted(glob.glob(f"{pattern}/X_scalar_*.npy"))
y_files = sorted(glob.glob(f"{pattern}/y_scalar_*.npy"))

X_chunks, y_chunks = [], []
for xf, yf in zip(X_files, y_files):
    X_chunks.append(np.load(xf))
    y_chunks.append((np.load(yf) > 0).astype(np.int8))

X_all = np.vstack(X_chunks)
y_all = np.concatenate(y_chunks)

print(f"Windows: {len(X_all)}  Features: {X_all.shape[1]}")
print(f"Labels:  {dict(zip(*np.unique(y_all, return_counts=True)))}")

Windows: 126468  Features: 22
Labels:  {np.int8(0): np.int64(121827), np.int8(1): np.int64(4641)}


### Preliminary Single-Sensor Results
**NOTE**: This uses a traditional stratified random split, which may not be sufficient for evaluation due to temporal nature of the data. Temporal leakage may or may not artificially boost evaluation metrics

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

print("=== RandomForest ===")
rf = RandomForestClassifier(n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
print(classification_report(y_test, rf.predict(X_test), target_names=["healthy", "fault"]))

print("=== XGBoost ===")
xgb = XGBClassifier(n_jobs=-1, random_state=42, eval_metric="logloss")
xgb.fit(X_train, y_train)
print(classification_report(y_test, xgb.predict(X_test), target_names=["healthy", "fault"]))

=== RandomForest ===
              precision    recall  f1-score   support

     healthy       1.00      1.00      1.00     24366
       fault       0.99      0.94      0.96       928

    accuracy                           1.00     25294
   macro avg       0.99      0.97      0.98     25294
weighted avg       1.00      1.00      1.00     25294

=== XGBoost ===
              precision    recall  f1-score   support

     healthy       1.00      1.00      1.00     24366
       fault       0.99      0.97      0.98       928

    accuracy                           1.00     25294
   macro avg       0.99      0.98      0.99     25294
weighted avg       1.00      1.00      1.00     25294

